Question 3 : What innovative approaches or methodologies are emerging in the field of coding?

Approach:

1 - Twitter (THIS NOTEBOOK)
2 - Dev.To
3 - Reddit

In [29]:
#Imports
import os.path as path
import os
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from nltk.chunk import ne_chunk

Using the prof's dataset "ChatGPT_Tweets_Dataset", I removed the columns 'tweet_id', 'Unnamed: 0', and 'users'. I dropped rows where the 'job_profiles' column had missing values, as I was interested in filtering later on only those tweets whose job_profile was known.
Before we had 3821843 rows, now we have 388283 rows.

In [37]:

twitter_df_path = path.join(os.getcwd(), "ChatGPT_Tweets_Dataset_Full.csv")
twitter_df = pd.read_csv(twitter_df_path)
del twitter_df['tweet_id']
del twitter_df["Unnamed: 0"]
del twitter_df["users"]
twitter_df = twitter_df.dropna(subset=['job_profiles'])

print(twitter_df.shape[0])

388283


I created a list of relevant keywords related to certain job profiles, such as "academic", "researcher", and "scientist", among others. Then, I constructed a regular expression pattern using these keywords. This pattern is used to filter `twitter_df` based on whether the 'job_profiles' column contains any of these keywords.
After filtering, we have 5326 rows in the dataframe.

In [82]:
# Define relevant keywords for relevant job_profiles to the question we have
relevant_keywords = ['academic', 'researcher', 'professor', 'scientist', 'computer science', 'data science',
                     'data scientist', 'software engineer', 'information technology', 'cybersecurity analyst',
                     'systems analyst', 'network engineer', 'database administrator', 'IT consultant', 'informatics',
                     'computer engineer', 'artificial intelligence', 'machine learning engineer', 'ML', 'DS']

# Create a regular expression pattern to match any of the relevant keywords
pattern = '|'.join(relevant_keywords)

# Filter the twitter_df based on whether 'job_profiles' contains any of the relevant keywords
filtered_jobs_df = twitter_df[twitter_df['job_profiles'].str.contains(pattern, case=False)]

print(filtered_jobs_df.shape[0])

5326


I filtered the dataset even more using keywords mentioning approaches, different technologies, metholodogies.
After filtering, we have 1416 rows in the dataframe.

In [84]:
# Filter tweets mentioning future, trends, methodologies, approaches, emerging technologies
keywords = ['future', 'trend', 'methodology', 'approach', 'emerging', 'emerged', 'rise', 'trending', 'technology', 'innovation', 'innovative',
            'progress', 'evolution', 'advance', 'strategy', 'technique', 'experiment', 'development', 'discovery',
            'breakthrough', 'revolution', 'coding', 'machine learning', 'AI', 'ML', 'automation', 'cloud computing', 'microservices', 
            'containerization', 'DevOps', 'low-code development', 'serverless computing', 'quantum computing']

filtered_tweets_df = filtered_jobs_df[filtered_jobs_df['original_text'].str.lower().str.contains('|'.join(keywords)) |
                        filtered_jobs_df['subjects'].str.lower().str.contains('|'.join(keywords)) |
                        filtered_jobs_df['topic'].str.lower().str.contains('|'.join(keywords))]


print("Rows")
print(filtered_tweets_df.shape[0])

Rows
1416


I used NLTK to tokenize and tag words for parts of speech, and also for named entity recognition. I set up a Counter to track frequencies of the approaches mentioned in tweets. I combined text from multiple columns (original_text, subjects, and topic) into a single string per tweet, tokenized this string, and tagged each word with its part of speech. I then extracted noun phrases to identify potential approaches.

I updated my frequency Counter with these noun phrases, ignoring named entities and filtering out common stop words and trivial entries (like single letters). I sorted the frequencies of these approaches in descending order and saved the result to a file.

In [91]:
# Tokenize and tag the words in the tweets using NLTK
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Initialize a Counter to store frequencies of innovative approaches or methodologies
approach_freq = Counter()

for index, row in filtered_tweets_df.iterrows():
    # Extract text from 'original_text' column
    tweet_text = row['original_text']
    
    # Additional columns you want to include
    subject_text = row['subjects']  # Replace 'other_column_name' with the actual column name
    topic_text = row['topic']  # Replace 'other_column_name' with the actual column name
    
    # Concatenate text from different columns
    full_text = tweet_text + ' ' + str(subject_text) + ' ' + str(topic_text)
    
    # Tokenize the concatenated text into words
    words = word_tokenize(full_text)
    
    # Tag the words with parts of speech
    tagged_words = nltk.pos_tag(words)
    
    # Extract noun phrases
    noun_phrases = [word for (word, pos) in tagged_words if pos.startswith('NN')]
    
    #Convert all noun phrases to lowercase
    noun_phrases_lower = [word.lower() for word in noun_phrases]
    
    # Update the Counter with lowercase noun phrases
    approach_freq.update(noun_phrases_lower)

# Remove named entities (persons) using named entity recognition (NER)
named_entities = ne_chunk(tagged_words)
filtered_noun_phrases = [word for word in noun_phrases_lower if not hasattr(word, 'label')]
    
# Remove common stop words and single-character words
stop_words = set(stopwords.words('english'))
approach_freq = {word: freq for word, freq in approach_freq.items() if word.lower() not in stop_words and len(word) > 1}

# Sort the approach frequencies in descending order
sorted_approach_freq = sorted(approach_freq.items(), key=lambda x: x[1], reverse=True)

with open("discoveryTwitter.txt", 'w') as file:
    # Write the header
    file.write("TWITTER Top innovative approaches or methodologies in coding from academic people discussing trends:\n")
    # Write the results to the file
    for approach, freq in sorted_approach_freq:
        file.write(f"{approach}: {freq} mentions\n")


[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/sara/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /Users/sara/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt to /Users/sara/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sara/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
